# Regression Analysis of penguin data
## Importing libraries and creating a NaN filtered dataset

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.axes as ax
import scipy
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pandas as pd

penguins = sns.load_dataset("penguins").dropna() # dopna removes rows with NaN data points

penguins.describe()


## Check models with variables of each non-categorical variable

In [ ]:

def plotPenguins(
    data: pd.Dataframe, fig: plt.Figure, axes: np.ndarray,
    rowIndex: int,  variables: list[str], name: str):

    sns.set()
    for i, var in enumerate(variables):
        model = smf.ols("body_mass_g ~" + var, data=data)
        result = model.fit()
        print(name + " " + var + "'s R²: " + str(result.rsquared))

        if (rowIndex == None):
            sns.scatterplot(
                data=data,
                x=var,
                y="body_mass_g",
                ax=axes[i],
                hue="sex",
                palette = {
                  "Male": "tab:blue",
                  "Female": "tab:pink",
                },
                style="species",
                legend=False
                )
            print(result.summary())
        else:
            sns.scatterplot(
                data=data,
                x=var,
                y="body_mass_g",
                ax=axes[rowIndex][i],
                hue="sex",
                palette = {
                  "Male": "tab:blue",
                  "Female": "tab:pink",
                },
                style="species",
                legend=False
                )

variables = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
fig, axes = plt.subplots(1,len(variables), figsize=(15,4))
filename = "CDV"
plotPenguins(penguins, fig, axes, None, variables, filename)

sns.scatterplot(
    data=penguins,
    x=variables[-1],
    y="body_mass_g",
    ax=axes[-1],
    hue="sex",
    palette = {
      "Male": "tab:blue",
      "Female": "tab:pink",
    },
    style="species"
    )
handles, labels = axes[-1].get_legend_handles_labels()
axes[-1].legend().remove()
fig.legend(handles, labels, bbox_to_anchor=(1.05,1), loc="upper left", borderaxespad=0)
fig.tight_layout()
fig.savefig(filename + ".jpg")

plt.show()


Flipper length seems to be the best variable for body mass, as the correlation seems highest, and the R² value closest to 1.
It can also be seen that both bill length and depth varies widely based on the species of penguin, which might be the reason for their low R² values. One can also see that the reason flipper length has a comparatively high R² value is that longer flipper length correlates with species and sex with higher body mass, which could be the real variables.
Notably, the bill depth has an inverse correlation with body mass. This is due to the Simpson's paradox, as we can see there's a positive correlation within 2 different groupings in the plots, but the adelie and chinstrap penguins have deeper bills that gentoo pegnuins despite their lower body mass. This causes the grouped model to inverse the covariance.
The estimated formula for body_mass_g ~ flipper_length_mm are:
$ŷ = 50.1533*x + -5872.0927$
In other words, $\beta_1^ = 50.1533$ and $\beta_0^ = -5872.0927$

## Filtering by species and sex

In [ ]:

penguinSpecies = ["Adelie", "Chinstrap", "Gentoo"]
sexes = ["Male", "Female"]
fig, axes = plt.subplots(len(sexes)*len(penguinSpecies),len(variables), figsize=(15,30))
for i, species in enumerate(penguinSpecies):
    for j, sex in enumerate(sexes):
        sortedPenguin = penguins[penguins["species"] == species]
        sortedPenguin = sortedPenguin[sortedPenguin["sex"] == sex]
        plotPenguins(sortedPenguin, fig, axes, 2*i+j, variables, species + "-" + sex)
        axes[2*i+j][0].set_ylabel(f"{species} - {sex}\nbody_mass_g")

fig.tight_layout()
plt.plot()
fig.savefig("CDV-filtered.jpg")


Here we can see that within the categories of sex within species, none of the dependency variables seem useful. This confirms the theory that flipper_length is a useful variable only due to its correlation with sex and species.

## More useful plots
### Linear plot
Despite this, we can see that a linear regression model fits the data well, and that we don't need to transform any variables.

In [ ]:

filename = "linearRegression"
fig, axes = plt.subplots(1,1, figsize=(5,5))
sns.regplot(x=penguins["flipper_length_mm"], y=penguins["body_mass_g"], ax=axes, line_kws={"color": "black"}, scatter_kws={"s": 2})
plt.plot()
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
print(model.fit().summary())


The variable flipper_length_mm being $\beta_1^ = 50.1533$ means that for every millimeter of flipper length extra measured on a penguin, it's likely for that the response (measured body mass) to be 50.1533g heavier. A penguin with longer flippers are likely to have a heavier body mass.
The estimated 95% confidence interval for $\beta_1^$ is CI=[47.123, 53.183]. With $\alpha=0.5$, it seems the variable does affect body_mass_g positively.
The low p value shows we can be quite certain of the linear connection.

### TA plot

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
sns.scatterplot(
                x=result.fittedvalues,
                y=result.resid,
                ax=axes,
                s=8,
                )
plt.ylabel("Residual")
plt.xlabel("Predicted value")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.show()


It can be seen here that there seems to be 2 clusters, and something that looks like a subtle curve.

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
sns.scatterplot(
                data=penguins,
                x=result.fittedvalues,
                y=result.resid,
                ax=axes,
                s=8,
                hue="sex",
                palette = {
                  "Male": "tab:blue",
                  "Female": "tab:pink",
                },
                style="species"
                )
plt.ylabel("Residual")
plt.xlabel("Predicted value")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left", borderaxespad=0)
plt.show()


The clusters aren't a sign of a poor model by themselves, but they do indicate there might be more to this graph. Earlier, we saw how the model doesn't hold up when filtering for the categories of sex and species, which we can tell here is the cause for the strange clustering.

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
ax = sns.scatterplot(
  x=result.fittedvalues,
  y=result.resid,
  ax=axes,
  s=8,
)

sns.regplot(
  x=result.fittedvalues,
  y= result.resid,
  lowess=True,
  scatter=False,
  color="red",
  ax=ax
)

plt.ylabel("Residual")
plt.xlabel("Predicted value")
ax.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.show()


Here we can see there's a clear curve in the TA plot. We'll see if it disappears when including more variables later.

### QQ plot

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
sm.qqplot(result.resid, line="45", fit=True)
plt.ylabel("Quantile in standardized residuals")
plt.xlabel("Quantile in normal distribution")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("QQ plot")
plt.show()


The QQ-plot can be seen here to seem fine, indicating a normal distribution. However, due to the issues with the TA-plot we still have to discard the model. The QQ-plot only shows the overall shape of the distribution, ignoring where it sits along the fitted values. Where these observations sit along the fitted values which we can observe in the TA-plot is what let us figure out the trends which make the model problematic.